In [1]:
import pandas as pd
from constants import DATA_PATH, EOS_FILE, SENTINEL_FILE

sentinel = pd.read_csv(DATA_PATH / SENTINEL_FILE)
eos = pd.read_csv(DATA_PATH / EOS_FILE)

In [2]:
sentinel = sentinel[sentinel['SM1 (%)'] != 50]
eos = eos[eos['SM1 (%)'] != 50]

In [3]:
from constants import X_cols_eos, X_cols_sentinel, y_col

X_sentinel = sentinel[X_cols_sentinel].values
X_eos = eos[X_cols_eos].values

y_sentinel = sentinel[y_col].values
y_eos = eos[y_col].values

In [4]:
import tensorflow as tf

I0000 00:00:1778444295.602555 2874623 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1778444295.630123 2874623 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


I0000 00:00:1778444296.314756 2874623 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [5]:
n_features = X_eos.shape[1]

models = {
#     "16, 1": tf.keras.Sequential([
#     # Input layer
#     tf.keras.Input(shape=(n_features, )),
#     tf.keras.layers.Dense(16, activation='relu'),
#     tf.keras.layers.Dense(1)
# ]),
#     "8, 1": tf.keras.Sequential([
#     # Input layer
#     tf.keras.Input(shape=(n_features, )),
#     tf.keras.layers.Dense(8, activation='relu'),
#     tf.keras.layers.Dense(1)
# ]),
#     "2, 1": tf.keras.Sequential([
#     # Input layer
#     tf.keras.Input(shape=(n_features, )),
#     tf.keras.layers.Dense(2, activation='relu'),
#     tf.keras.layers.Dense(1)
# ]),
#     "4, 1": tf.keras.Sequential([
#     # Input layer
#     tf.keras.Input(shape=(n_features, )),
#     tf.keras.layers.Dense(4, activation='relu'),
#     tf.keras.layers.Dense(1)
# ]),
    "16, Dropout, 8, Dropout": tf.keras.Sequential([
    # Input layer
    tf.keras.Input(shape=(n_features, )),
    tf.keras.layers.Dense(16, activation='relu'),
    tf.keras.layers.Dropout(0.09),
    tf.keras.layers.Dense(8, activation='relu'),
    tf.keras.layers.Dropout(0.09),
    tf.keras.layers.Dense(1)
]),
    "16, Dropout": tf.keras.Sequential([
    # Input layer
    tf.keras.Input(shape=(n_features, )),
    tf.keras.layers.Dense(16, activation='relu'),
    tf.keras.layers.Dropout(0.1),
    tf.keras.layers.Dense(1)
])
}

I0000 00:00:1778444297.129798 2874623 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 6157 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4060 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.9


In [6]:
from model_experiments import PredictionIntervalEstimation

tf.keras.backend.clear_session()

eos_results = {}

for param_string, model in models.items():
    optimizer = tf.keras.optimizers.Adam(learning_rate=0.0001)

    exp = PredictionIntervalEstimation(X_eos, y_eos, satellite="EOS-04")
    results = exp.run_experiment(model, model_param_string=param_string, optimizer=optimizer, epochs=1000)
    eos_results[param_string] = results

Results → /home/lmaosid/Desktop/major/experiments/classification_new_data/output/pi_estimation_uncensored


Upper model:   0%|          | 0/1000 [00:00<?, ?epoch/s]

I0000 00:00:1778444298.212144 2874720 service.cc:153] XLA service 0x7d24cc0322d0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1778444298.212158 2874720 service.cc:161]   StreamExecutor [0]: NVIDIA GeForce RTX 4060 Laptop GPU, Compute Capability 8.9 (Driver: 13.2.0; Runtime: 12.4.0; Toolkit: 12.5.0; DNN: 9.3.0)
I0000 00:00:1778444298.223151 2874720 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1778444298.290407 2874720 cuda_dnn.cc:461] Loaded cuDNN version 90300
I0000 00:00:1778444298.317063 2874720 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_1529__.8


I0000 00:00:1778444299.352669 2874720 device_compiler.h:208] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.
I0000 00:00:1778444299.465030 2874716 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_1529__.8


Upper model:   0%|          | 0/1000 [00:02<?, ?epoch/s, loss=17.9491, val_loss=16.4972]

...

6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step


1/6 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step

6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step

6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step


16, Dropout: {
    "val": {
        "PICP": 0.960674,
        "MPIW": 31.458364
    },
    "test": {
        "PICP": 0.944134,
        "MPIW": 32.619976
    }
}


In [7]:
from constants import OUTPUT_PATH
import json

with open(OUTPUT_PATH / "pi_estimation_uncensored" / "EOS-04_metrics.json", "w") as f:
    json.dump(eos_results, f, indent=4)

In [8]:
from model_experiments import PredictionIntervalEstimation
sentinel_results = {}

for param_string, model in models.items():
    tf.keras.backend.clear_session()
    optimizer = tf.keras.optimizers.Adam(learning_rate=0.0001)

    exp = PredictionIntervalEstimation(X_sentinel, y_sentinel, satellite="Sentinel-1")
    results = exp.run_experiment(model, model_param_string=param_string, optimizer=optimizer, epochs=1000)
    sentinel_results[param_string] = results

Results → /home/lmaosid/Desktop/major/experiments/classification_new_data/output/pi_estimation_uncensored


Upper model:   0%|          | 0/1000 [00:00<?, ?epoch/s]

I0000 00:00:1778444344.798047 2874716 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_36382__.8


I0000 00:00:1778444345.534821 2874714 dot_merger.cc:481] Merging Dots in computation: a_inference_one_step_on_data_36382__.8


Upper model:   0%|          | 0/1000 [00:02<?, ?epoch/s, loss=19.5317, val_loss=19.2601]

...

1/5 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step

5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 


1/5 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step

5/5 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step 


16, Dropout: {
    "val": {
        "PICP": 0.948905,
        "MPIW": 40.062069
    },
    "test": {
        "PICP": 0.970803,
        "MPIW": 40.822487
    }
}


In [9]:
from constants import OUTPUT_PATH
import json

with open(OUTPUT_PATH / "pi_estimation_uncensored" / "Sentinel-1_metrics.json", "w") as f:
    json.dump(sentinel_results, f, indent=4)